# Mantenimiento Predictivo — KAESER CSDX165

**Proyecto Final · Curso de Inteligencia Artificial Industrial**  
**Autor:** Felipe Soto

---

### Problema
Predecir la advertencia de diferencial de presión en filtros de aceite (**0011 W**)  
y filtros de aire (**0013 W**) antes de que ocurra, a partir de datos sensoriales  
registrados a 1 Hz por el controlador SIGMA CONTROL 2 (archivos `.dat`).

### Estado del Notebook

| Commit | Secciones | Estado |
|--------|-----------|--------|
| **C1** | Imports · Configuración · Carga DAT · Parser de eventos | ✅ *este* |
| C2 | Extracción de canales · Features · Labels | 🔲 |
| C3 | Ensamblado del dataset | 🔲 |
| C4 | Modelos (DT · RF · MLP) | 🔲 |
| C5 | Evaluación · Interpretación · Conclusiones | 🔲 |

In [ ]:
# ── 0.1  Imports ──────────────────────────────────────────────────────────────
import os, re, glob, struct
from datetime import datetime

import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt   # usado desde C2 en adelante

# Semilla de reproducibilidad — se fija aquí, nunca se sobreescribe
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print('Imports OK')

---
## 0.2  Configuración

Todas las rutas, constantes y esquemas de columnas se definen aquí.  
El resto del notebook **no repite** estos valores.

In [ ]:
# ── Rutas base ────────────────────────────────────────────────────────────────
ROOT      = r'C:\Users\Felipe\Desktop\Industrias\2026\IA\proyecto\equipos'
PROJ_ROOT = r'C:\Users\Felipe\Desktop\Industrias\2026\IA\proyecto'

# Fragmentos de ruta intermedios (mantienen COMPRESSORS legible)
_d1  = os.path.join(ROOT, 'Compresor CSDX165 Aire Industrial N\u00ba1', 'CSDX165_INDN\u00ba1')
_d2  = os.path.join(ROOT, 'Compresor CSDX165 Aire Industrial N\u00ba2', 'CSDX165_INDN\u00ba2')
_bf1 = 'BF_101803.1_1171_2026-05-27T09.48.42-04.00'
_bf2 = 'BF_101803.2_1001_2026-05-27T09.54.00-04.00'

COMPRESSORS = {
    'INDN\u00ba1': {
        'dat_root':    os.path.join(_d1, 'datarecorder'),
        'log_path':    os.path.join(_d1, _bf1, 'reports_lang', 'CompressorMsgs.txt'),
        'target_code': '0011',
        'target_cat':  'W',
        'dat_start':   datetime(2026, 3, 18),
        'dat_end':     datetime(2026, 5, 27),
    },
    'INDN\u00ba2': {
        'dat_root':    os.path.join(_d2, 'datarecorder'),
        'log_path':    os.path.join(_d2, _bf2, 'reports_lang', 'CompressorMsgs.txt'),
        'target_code': '0013',
        'target_cat':  'W',
        'dat_start':   datetime(2025, 6, 23),
        'dat_end':     datetime(2026, 5, 27),
    },
}
print('Paths OK')

In [ ]:
# ── Formato binario DAT ───────────────────────────────────────────────────────
DAT_VALID_SIZE    = 954_664   # bytes — archivos de otro tamaño se descartan
DAT_DATA_OFFSET   = 4264      # inicio del bloque de datos (tras la cabecera)
DAT_REC_SIZE      = 264       # bytes por registro
DAT_N_RECORDS     = 3600      # registros por archivo (1 Hz x 3600 s = 1 h)
DAT_N_CHANNELS    = 130       # canales int16 por registro
DAT_MISSING_INT16 = -32768    # valor centinela -> NaN

# ── Canales seleccionados (Tier-1 + Tier-2) ───────────────────────────────────
# Confirmados por auditoría de canal (_channel_map.py / _channel_live_check.py)
CH = {
    'cs':         0,   # Compressor status    — bitmask (en marcha/cargado)
    'pressure':   4,   # System pressure      — x0.01 -> bar
    'oil_temp':   6,   # Oil separator temp.  — x0.01 -> grados C
    'inlet_temp': 9,   # Inlet temperature    — x0.1  -> grados C  (proxy ambiental)
    'speed_sp':  12,   # Compressor speed SP  — x10   -> RPM       (proxy de carga)
    # Tier-2
    'fan_speed': 38,   # Oil cooler fan speed — x10   -> RPM
    'etm_valve': 44,   # ETM valve pos. SP    — x0.1  -> %
    'etm_fan_sp':45,   # ETM fan speed SP     — x0.1  -> %
    'etm_ctrl':  47,   # ETM dT ctrl. var.    — x0.1  -> %
}

CH_SCALE = {
    'cs': 1.0,   'pressure': 0.01, 'oil_temp': 0.01,
    'inlet_temp': 0.1, 'speed_sp': 10.0, 'fan_speed': 10.0,
    'etm_valve': 0.1,  'etm_fan_sp': 0.1, 'etm_ctrl': 0.1,
}

CH_UNIT = {
    'cs': 'bitmask', 'pressure': 'bar',  'oil_temp': 'C',
    'inlet_temp': 'C', 'speed_sp': 'RPM', 'fan_speed': 'RPM',
    'etm_valve': '%',  'etm_fan_sp': '%', 'etm_ctrl': '%',
}

# Orden de extraccion — define el orden de columnas en los arrays raw
CH_NAMES        = list(CH.keys())             # ['cs', 'pressure', ...]
EXTRACT_INDICES = [CH[k] for k in CH_NAMES]   # [0, 4, 6, 9, 12, 38, 44, 45, 47]

# ── Constantes de etiquetado (usadas desde la Seccion 2) ─────────────────────
PREDICTION_HORIZON_H = 24   # horas previas al onset = etiqueta positiva
RECOVERY_EXCLUSION_H =  4   # horas post-despeje excluidas del entrenamiento
EPISODE_GAP_DAYS     =  7   # brecha minima (dias) entre episodios distintos

ZONE_NEGATIVE = 'negative'
ZONE_PRE_WARN = 'pre_warning'
ZONE_ACTIVE   = 'active_warning'
ZONE_RECOVERY = 'recovery'
TARGET_COL    = 'label_extended'

# ── Columnas del modelo (forward declaration — pobladas en la Seccion 2) ─────
FEATURE_COLS = [
    'p_mean_1h',        'p_std_1h',          'p_mean_24h',        'p_trend_24h',
    'oil_temp_mean_1h', 'oil_temp_max_1h',   'oil_temp_mean_24h', 'oil_temp_trend_24h',
    'inlet_temp_mean_24h',
    'speed_mean_1h',    'speed_mean_24h',    'speed_trend_24h',
    'load_frac_1h',     'load_frac_24h',     'starts_24h',
    'fan_speed_mean_1h','etm_valve_mean_1h', 'etm_fan_sp_mean_1h',
    'month_sin',        'month_cos',         'hour_sin',          'hour_cos',
    'hours_loaded_since_clear',
]  # 23 features

print(f'Config OK  —  {len(FEATURE_COLS)} features declaradas,  {len(CH)} canales seleccionados')

---
## 1  Utilidades de Carga DAT

Dos funciones de bajo nivel. Sin lógica de features ni escalado.

| Función | Entrada | Salida |
|---------|---------|--------|
| `list_dat_files(dat_root)` | directorio del recorder | lista ordenada de rutas `.dat` válidas |
| `read_dat_hour(filepath)` | ruta de un `.dat` | `(hour_utc, raw_array)` |

In [ ]:
# ── 1.1  list_dat_files ───────────────────────────────────────────────────────
def list_dat_files(dat_root):
    """
    Return sorted list of valid DAT file paths under dat_root.
    Rejects files whose size differs from DAT_VALID_SIZE (truncated/partial).
    """
    files = sorted(
        glob.glob(os.path.join(dat_root, '**', 'mcs_*.dat'), recursive=True)
    )
    return [f for f in files if os.path.getsize(f) == DAT_VALID_SIZE]


# ── 1.2  read_dat_hour ────────────────────────────────────────────────────────
def read_dat_hour(filepath):
    """
    Read one DAT file.

    Returns
    -------
    hour_utc : datetime
        UTC time of record 0 (= file-hour start).
    raw : ndarray, shape (3600, len(EXTRACT_INDICES)), float64
        Raw int16 channel values; sentinel (-32768) replaced with NaN.
        Column order: CH_NAMES  /  EXTRACT_INDICES.
        Scale factors are NOT applied here.
    """
    with open(filepath, 'rb') as fh:
        data = fh.read()

    # UTC timestamp from first record (uint32 LE at DATA_OFFSET)
    ts0      = struct.unpack_from('<I', data, DAT_DATA_OFFSET)[0]
    hour_utc = datetime.utcfromtimestamp(ts0)

    # Read every record as int16 words.
    # Layout per 264-byte record: [ts_lo, ts_hi, ch0, ch1, ..., ch129]
    #   264 bytes / 2 = 132 int16 per record
    #   cols 0-1  : timestamp (two int16 = one uint32)
    #   cols 2-131: 130 channel values
    flat = np.frombuffer(
        data, dtype='<i2',
        offset=DAT_DATA_OFFSET,
        count=DAT_N_RECORDS * (DAT_REC_SIZE // 2)
    ).reshape(DAT_N_RECORDS, DAT_REC_SIZE // 2)   # (3600, 132)

    ch_all = flat[:, 2:].astype(np.float64)        # (3600, 130)
    ch_all[ch_all == DAT_MISSING_INT16] = np.nan

    return hour_utc, ch_all[:, EXTRACT_INDICES]


print('DAT utilities OK')

---
## 2  Utilidades de Carga del Log de Eventos

| Función | Entrada | Salida |
|---------|---------|--------|
| `parse_event_log(log_path, code, cat)` | ruta + filtro de código/categoría | `df_events` |
| `build_episode_table(df_events, start, end)` | `df_events` + ventana DAT | `df_episodes` |

In [ ]:
# ── 2.1  parse_event_log ──────────────────────────────────────────────────────
_HEADER_RE = re.compile(
    r'^(\d{4})\s+([OWA])\s+([cga])\s+'
    r'(\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2})([-+]\d{2}):?(\d{2})'
)

def parse_event_log(log_path, target_code, target_cat):
    """
    Parse CompressorMsgs.txt and return a DataFrame of target events.

    Parameters
    ----------
    log_path    : str  — path to CompressorMsgs.txt
    target_code : str  — 4-digit event code, e.g. '0011'
    target_cat  : str  — category char: 'O', 'W', or 'A'

    Returns
    -------
    DataFrame columns:
        timestamp   datetime  — local time as recorded (no UTC conversion)
        state       str       — 'c' (onset) | 'g' (clear)
        tz_offset_h int       — signed hours offset from log line
    Sorted ascending by timestamp.
    Exact (timestamp, state) duplicates removed.
    """
    rows = []
    with open(log_path, encoding='utf-8', errors='replace') as fh:
        for line in fh:
            m = _HEADER_RE.match(line.strip())
            if not m:
                continue
            if m.group(1) != target_code or m.group(2) != target_cat:
                continue
            dt   = datetime.strptime(m.group(4), '%Y-%m-%dT%H:%M:%S')
            tz_h = int(m.group(5))
            rows.append({'timestamp': dt, 'state': m.group(3), 'tz_offset_h': tz_h})

    if not rows:
        return pd.DataFrame(columns=['timestamp', 'state', 'tz_offset_h'])

    return (
        pd.DataFrame(rows)
          .sort_values('timestamp')
          .drop_duplicates(subset=['timestamp', 'state'])
          .reset_index(drop=True)
    )


print('parse_event_log OK')

In [ ]:
# ── 2.2  build_episode_table ──────────────────────────────────────────────────
def build_episode_table(df_events, dat_start, dat_end):
    """
    Group events into maintenance episodes (EPISODE_GAP_DAYS threshold).
    Returns only episodes whose first onset falls inside [dat_start, dat_end].

    Parameters
    ----------
    df_events : DataFrame from parse_event_log()
    dat_start : datetime — first date of DAT coverage
    dat_end   : datetime — last date of DAT coverage

    Returns
    -------
    DataFrame columns:
        episode_id    int
        first_onset   datetime
        last_clear    datetime  (NaT if episode never cleared in log)
        n_onsets      int
        in_dat        bool
        duration_days float     (NaN if last_clear is NaT)
    Only in_dat == True rows returned.
    """
    onsets = df_events[df_events['state'] == 'c'].reset_index(drop=True)
    clears = df_events[df_events['state'] == 'g'].reset_index(drop=True)

    _empty = pd.DataFrame(
        columns=['episode_id', 'first_onset', 'last_clear',
                 'n_onsets', 'in_dat', 'duration_days']
    )
    if onsets.empty:
        return _empty

    # ── Cluster onsets by EPISODE_GAP_DAYS gap ────────────────────────────────
    ep_groups = []
    current   = [onsets.iloc[0]['timestamp']]
    for i in range(1, len(onsets)):
        gap_s = (
            onsets.iloc[i]['timestamp'] - onsets.iloc[i - 1]['timestamp']
        ).total_seconds()
        if gap_s > EPISODE_GAP_DAYS * 86_400:
            ep_groups.append(current)
            current = []
        current.append(onsets.iloc[i]['timestamp'])
    ep_groups.append(current)

    # ── Build one row per episode ─────────────────────────────────────────────
    rows = []
    for ep_id, ep_ts in enumerate(ep_groups, 1):
        first_onset = ep_ts[0]
        # Last clear: any 'g' after this episode's first onset
        #             and before the next episode's first onset
        upper  = ep_groups[ep_id][0] if ep_id < len(ep_groups) else pd.Timestamp.max
        mask   = (clears['timestamp'] > first_onset) & (clears['timestamp'] < upper)
        last_clear = clears.loc[mask, 'timestamp'].max() if mask.any() else pd.NaT
        dur = (
            (last_clear - first_onset).total_seconds() / 86_400
            if pd.notna(last_clear) else np.nan
        )
        rows.append({
            'episode_id':    ep_id,
            'first_onset':   first_onset,
            'last_clear':    last_clear,
            'n_onsets':      len(ep_ts),
            'in_dat':        dat_start <= first_onset <= dat_end,
            'duration_days': round(dur, 2) if pd.notna(last_clear) else np.nan,
        })

    df_eps = pd.DataFrame(rows)
    return df_eps[df_eps['in_dat']].reset_index(drop=True)


print('build_episode_table OK')

---
## Verificación — Commit 1

Comprueba que las utilidades de carga producen los conteos y marcas de tiempo  
esperados, confirmados en la auditoría previa.

| Compressor | DAT files | DAT start | DAT end | Onsets in-DAT | Episodios in-DAT |
|------------|-----------|-----------|---------|---------------|------------------|
| INDNº1 | ~1 650 | 2026-03-18 | 2026-05-27 | 17 | 1 |
| INDNº2 | ~8 045 | 2025-06-23 | 2026-05-27 | 19 | 4 |

In [ ]:
SEP  = '=' * 70
DSEP = '-' * 70

print(SEP)
print('  COMMIT 1  -  VERIFICACION')
print(SEP)

for comp_id, cfg in COMPRESSORS.items():
    code = cfg['target_code']
    cat  = cfg['target_cat']

    print(f'\n{DSEP}')
    print(f'  Compressor   : {comp_id}   (evento objetivo: {code} {cat})')
    print(DSEP)

    # ── DAT files ─────────────────────────────────────────────────────────────
    files = list_dat_files(cfg['dat_root'])
    print(f'  DAT files    : {len(files):,}')

    if files:
        t_first, _ = read_dat_hour(files[0])
        t_last,  _ = read_dat_hour(files[-1])
        print(f'  DAT first    : {t_first}  UTC')
        print(f'  DAT last     : {t_last}  UTC')
    else:
        print('  ADVERTENCIA  : no se encontraron archivos DAT validos.')

    # ── Event log ─────────────────────────────────────────────────────────────
    df_ev   = parse_event_log(cfg['log_path'], code, cat)
    n_came  = (df_ev['state'] == 'c').sum()
    n_gone  = (df_ev['state'] == 'g').sum()
    print(f'\n  Eventos {code} {cat} : {len(df_ev):,} total'
          f'  ({n_came} onsets  /  {n_gone} clearances)')

    # ── Episodes ──────────────────────────────────────────────────────────────
    df_ep = build_episode_table(df_ev, cfg['dat_start'], cfg['dat_end'])
    print(f'  Episodios in-DAT : {len(df_ep)}')

    if not df_ep.empty:
        cols = ['episode_id', 'first_onset', 'last_clear', 'n_onsets', 'duration_days']
        print()
        print(df_ep[cols].to_string(index=False))

print(f'\n{SEP}')
print('  OK  -  Commit 1 completo.  Pipeline foundation lista.')
print(SEP)